## Análise inicial - Projeto do Ciclo Básico InsperData - Grupo 2
#### Integrantes:  Laura, Matias e Vinícius

Importação das bibliotecas necessárias:

In [1]:
from nba_api.stats.endpoints import leaguegamelog
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


In [2]:
# Pegando os dados da última temporada
gamelog = leaguegamelog.LeagueGameLog(season="2024-25", season_type_all_star="Regular Season")
dataframe = gamelog.get_data_frames()[0] #lista com 1 df com os filtros selecionados, pegar esse único termo

dataframe.head()

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE
0,22024,1610612738,BOS,Boston Celtics,0022400061,2024-10-22,BOS vs. NYK,W,240,48,...,29,40,33,6,3,4,15,132,23,1
1,22024,1610612750,MIN,Minnesota Timberwolves,0022400062,2024-10-22,MIN @ LAL,L,240,35,...,35,47,17,4,1,16,22,103,-7,1
2,22024,1610612747,LAL,Los Angeles Lakers,0022400062,2024-10-22,LAL vs. MIN,W,240,42,...,31,46,22,7,8,7,22,110,7,1
3,22024,1610612752,NYK,New York Knicks,0022400061,2024-10-22,NYK @ BOS,L,240,43,...,29,34,20,2,3,12,12,109,-23,1
4,22024,1610612744,GSW,Golden State Warriors,0022400072,2024-10-23,GSW @ POR,W,240,48,...,42,57,38,13,5,18,27,140,36,1


In [3]:
print(dataframe.columns)

Index(['SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID',
       'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M',
       'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST',
       'STL', 'BLK', 'TOV', 'PF', 'PTS', 'PLUS_MINUS', 'VIDEO_AVAILABLE'],
      dtype='str')


In [4]:
df = dataframe[['GAME_ID',
        'TEAM_ABBREVIATION', 
        'TEAM_NAME', 
        'GAME_DATE', 
        'MATCHUP', # se for @ é fora de casa pro X @ x; se for vs é em casa pra X vs x;
        'WL', # TARGET
        'PTS', 
        'FG_PCT', # porcentagem de aproveitamento
        'FG3_PCT', # porcentagem de aproveitamento de 3 
        'FT_PCT', # porcentagem de aproveitamento de lance livre
        'PLUS_MINUS', # saldo dos pontos do jogo
        'OREB', # rebotes ofensivos (segunda chance de ataque)
        'AST', # assistências
        'TOV']].copy() # erros ou perdas de posse

df.dtypes


TEAM_ABBREVIATION        str
TEAM_NAME                str
GAME_DATE                str
MATCHUP                  str
WL                       str
PTS                    int64
FG_PCT               float64
FG3_PCT              float64
FT_PCT               float64
PLUS_MINUS             int64
OREB                   int64
AST                    int64
TOV                    int64
dtype: object

In [5]:
# atualizando tipos de variáveis
df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE']) # transformando em data

# retorna 1 se o jogo é em casa para o primeiro time do MATCHUP; 0 se é fora de casa
df['HOME_AWAY'] = df['MATCHUP'].apply(lambda x: 1 if 'vs' in x else 0)
df['HOME_AWAY'] = df['HOME_AWAY'].astype('category')
df['WL'] = df['WL'].astype('category')

# extraindo a sigla do adversário a partir do MATCHUP
# exemplos: "BOS vs. NYK" -> NYK; "BOS @ NYK" -> NYK
df['OPPONENT_ABBREVIATION'] = df['MATCHUP'].str.extract(r'(?:vs\.?|@)\s+([A-Z]{2,3})', expand=False)

df.dtypes


TEAM_ABBREVIATION               str
TEAM_NAME                       str
GAME_DATE            datetime64[us]
MATCHUP                         str
WL                         category
PTS                           int64
FG_PCT                      float64
FG3_PCT                     float64
FT_PCT                      float64
PLUS_MINUS                    int64
OREB                          int64
AST                           int64
TOV                           int64
HOME_AWAY                  category
dtype: object

In [6]:
# organizando o df para que os jogos de times diferentes não fiquem misturados e a ordem das datas esteja coerente
df = df.sort_values(['TEAM_ABBREVIATION', 'GAME_DATE', 'GAME_ID']).copy()


In [ ]:

# ------------------------------------------------------------
# Cálculo do descanso do time E do adversário
# ------------------------------------------------------------
# Como o LeagueGameLog possui duas linhas por partida, uma para cada time,
# primeiro calculamos o descanso de cada time individualmente e depois
# fazemos um merge para trazer o descanso do oponente na mesma partida.

# coluna do último jogo para calcular os dias de descanso, agrupados por time
df['PREV_GAME'] = df.groupby('TEAM_ABBREVIATION')['GAME_DATE'].shift(1)
df['REST_DAYS_NUM'] = (df['GAME_DATE'] - df['PREV_GAME']).dt.days - 1

def classificar_descanso(x):
    if pd.isna(x):
        return np.nan
    elif x <= 0:
        return '0'
    elif x == 1:
        return '1'
    else:
        return '2+'

# descanso do próprio time em categorias: 0, 1 ou 2+
df['REST_DAYS'] = df['REST_DAYS_NUM'].apply(classificar_descanso)

# tabela auxiliar com o descanso de cada time em cada jogo
opponent_rest = df[['GAME_ID', 'TEAM_ABBREVIATION', 'REST_DAYS', 'REST_DAYS_NUM']].copy()
opponent_rest = opponent_rest.rename(columns={
    'TEAM_ABBREVIATION': 'OPPONENT_ABBREVIATION',
    'REST_DAYS': 'OPP_REST_DAYS',
    'REST_DAYS_NUM': 'OPP_REST_DAYS_NUM'
})

# trazendo para cada linha o descanso do adversário naquela mesma partida
df = df.merge(
    opponent_rest,
    on=['GAME_ID', 'OPPONENT_ABBREVIATION'],
    how='left'
)

# removendo jogos em que o time ou o adversário ainda não tinha jogo anterior na temporada
df = df.dropna(subset=['REST_DAYS', 'OPP_REST_DAYS']).copy()

# mantendo a ordem lógica das categorias
rest_categories = ['0', '1', '2+']
df['REST_DAYS'] = pd.Categorical(df['REST_DAYS'], categories=rest_categories, ordered=True)
df['OPP_REST_DAYS'] = pd.Categorical(df['OPP_REST_DAYS'], categories=rest_categories, ordered=True)

# variável detalhada: descanso do time vs descanso do oponente
# exemplo: "0 vs 2+" = time em back-to-back contra adversário com 2+ dias de descanso
df['REST_MATCHUP'] = df['REST_DAYS'].astype(str) + ' vs ' + df['OPP_REST_DAYS'].astype(str)

rest_matchup_order = [
    '0 vs 0', '0 vs 1', '0 vs 2+',
    '1 vs 0', '1 vs 1', '1 vs 2+',
    '2+ vs 0', '2+ vs 1', '2+ vs 2+'
]
df['REST_MATCHUP'] = pd.Categorical(df['REST_MATCHUP'], categories=rest_matchup_order, ordered=True)

# variável complementar: vantagem relativa de descanso
# valor negativo = time analisado tem menos descanso que o adversário
# valor zero = ambos têm o mesmo descanso
# valor positivo = time analisado tem mais descanso que o adversário
rest_to_num = {'0': 0, '1': 1, '2+': 2}
df['REST_DAYS_NUM_CAT'] = df['REST_DAYS'].astype(str).map(rest_to_num)
df['OPP_REST_DAYS_NUM_CAT'] = df['OPP_REST_DAYS'].astype(str).map(rest_to_num)
df['REST_ADVANTAGE'] = df['REST_DAYS_NUM_CAT'] - df['OPP_REST_DAYS_NUM_CAT']

# variável simplificada para os gráficos de barras
# A ideia é reduzir as 9 combinações para 4 grupos:
# 1) descanso igual
# 2) 0 vs 1
# 3) 0 vs 2+
# 4) 1 vs 2+
#
# Nos jogos com descanso diferente, os gráficos de desempenho serão lidos
# pela perspectiva do time com MENOS descanso.
def classificar_comparativo_descanso(row):
    team_rest = str(row['REST_DAYS'])
    opp_rest = str(row['OPP_REST_DAYS'])

    if team_rest == opp_rest:
        return 'Descanso igual'

    ordered_pair = sorted([team_rest, opp_rest], key=lambda x: rest_to_num[x])
    return f'{ordered_pair[0]} vs {ordered_pair[1]}'

rest_comparison_order = ['Descanso igual', '0 vs 1', '0 vs 2+', '1 vs 2+']
df['REST_COMPARISON_GROUP'] = df.apply(classificar_comparativo_descanso, axis=1)
df['REST_COMPARISON_GROUP'] = pd.Categorical(
    df['REST_COMPARISON_GROUP'],
    categories=rest_comparison_order,
    ordered=True
)

df.head()


In [8]:
# coluna com 1 para os jogos em que o time analisado está em back-to-back
df['BACK_TO_BACK'] = (df['REST_DAYS'] == '0').astype(int)

# coluna opcional: back-to-back contra adversário mais descansado
df['B2B_VS_RESTED_OPP'] = ((df['REST_DAYS'] == '0') & (df['OPP_REST_DAYS'].isin(['1', '2+']))).astype(int)

df.head()


,TEAM_ABBREVIATION,TEAM_NAME,GAME_DATE,MATCHUP,WL,PTS,FG_PCT,FG3_PCT,FT_PCT,PLUS_MINUS,OREB,AST,TOV,HOME_AWAY,PREV_GAME,REST_DAYS,BACK_TO_BACK
32,ATL,Atlanta Hawks,2024-10-25,ATL vs. CHA,W,125,0.481,0.368,0.868,5,7,25,14,1,2024-10-23,1,0
77,ATL,Atlanta Hawks,2024-10-27,ATL @ OKC,L,104,0.396,0.323,0.759,-24,17,24,20,0,2024-10-25,1,0
95,ATL,Atlanta Hawks,2024-10-28,ATL vs. WAS,L,119,0.481,0.375,0.722,-2,6,32,16,1,2024-10-27,0,1
112,ATL,Atlanta Hawks,2024-10-30,ATL @ WAS,L,120,0.474,0.308,0.857,-13,12,28,17,0,2024-10-28,1,0
142,ATL,Atlanta Hawks,2024-11-01,ATL vs. SAC,L,115,0.446,0.380,0.778,-8,8,30,13,1,2024-10-30,1,0


In [9]:
# transformando vitória/derrota em variável binária: vitória = 1, derrota = 0
df['WL'] = df['WL'].map({'W': 1, 'L': 0}).astype(int)
df.head()


,TEAM_ABBREVIATION,TEAM_NAME,GAME_DATE,MATCHUP,WL,PTS,FG_PCT,FG3_PCT,FT_PCT,PLUS_MINUS,OREB,AST,TOV,HOME_AWAY,PREV_GAME,REST_DAYS,BACK_TO_BACK
32,ATL,Atlanta Hawks,2024-10-25,ATL vs. CHA,1,125,0.481,0.368,0.868,5,7,25,14,1,2024-10-23,1,0
77,ATL,Atlanta Hawks,2024-10-27,ATL @ OKC,0,104,0.396,0.323,0.759,-24,17,24,20,0,2024-10-25,1,0
95,ATL,Atlanta Hawks,2024-10-28,ATL vs. WAS,0,119,0.481,0.375,0.722,-2,6,32,16,1,2024-10-27,0,1
112,ATL,Atlanta Hawks,2024-10-30,ATL @ WAS,0,120,0.474,0.308,0.857,-13,12,28,17,0,2024-10-28,1,0
142,ATL,Atlanta Hawks,2024-11-01,ATL vs. SAC,0,115,0.446,0.380,0.778,-8,8,30,13,1,2024-10-30,1,0


In [ ]:
# removendo jogos a partir do dia 20 de março de 2025, quando começam as eliminações matemáticas dos playoffs e aumenta o incentivo ao tanking

In [ ]:
df.to_csv('nba_descanso_com_oponente_grupos_simplificados.csv', index=False)


### Análises iniciais

In [ ]:

# distribuição dos jogos por agrupamento simplificado de descanso
# Para contar jogos reais, e não duas linhas por partida, usamos uma linha por GAME_ID.
# Nos jogos com descanso diferente, mantemos apenas a linha do time com menor descanso.
df_barras_jogos = (
    df[df['REST_ADVANTAGE'] <= 0]
    .drop_duplicates(subset=['GAME_ID'])
    .copy()
)

plt.figure(figsize=(10,6))
sns.countplot(
    x='REST_COMPARISON_GROUP',
    data=df_barras_jogos,
    stat='percent',
    palette='muted',
    order=rest_comparison_order
)
plt.title('Distribuição dos jogos por diferença de descanso', loc='center', fontsize=15)
plt.xlabel('Comparação de descanso entre os times', fontsize=13, loc='center')
plt.ylabel('Percentual dos jogos (%)', fontsize=13, loc='center')
plt.xticks(rotation=0)
plt.grid(visible=True, alpha=0.7, ls='--', axis='y')
plt.tight_layout()
plt.savefig('grafico_descanso_grupos_simplificados.jpg')
plt.show()


In [ ]:

# Desempenho por agrupamento simplificado de descanso
# Nos grupos 0 vs 1, 0 vs 2+ e 1 vs 2+, o resultado é calculado
# pela perspectiva do time com MENOS descanso.
# No grupo "Descanso igual", os dois times são mantidos como referência neutra.
df_barras_desempenho = df[df['REST_ADVANTAGE'] <= 0].copy()

plt.figure(figsize=(10,6))
sns.barplot(
    x='REST_COMPARISON_GROUP',
    y='WL',
    data=df_barras_desempenho,
    palette='muted',
    errorbar=None,
    order=rest_comparison_order
)
plt.title('Taxa de vitória por diferença de descanso', fontsize=15, loc='center')
plt.xlabel('Comparação de descanso entre os times', fontsize=13, loc='center')
plt.ylabel('Proporção de vitórias', fontsize=13, loc='center')
plt.ylim(0,1)
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('vitoria_por_descanso_grupos_simplificados.jpg')
plt.show()


In [ ]:

# Saldo médio por agrupamento simplificado de descanso
# Nos grupos com descanso diferente, o saldo é calculado pela perspectiva
# do time com MENOS descanso.
plt.figure(figsize=(10,6))
sns.barplot(
    x='REST_COMPARISON_GROUP',
    y='PLUS_MINUS',
    data=df_barras_desempenho,
    palette='muted',
    errorbar=None,
    order=rest_comparison_order
)
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.title('Saldo médio por diferença de descanso', fontsize=15, loc='center')
plt.xlabel('Comparação de descanso entre os times', fontsize=13, loc='center')
plt.ylabel('Saldo médio do jogo', fontsize=13, loc='center')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.7)
plt.tight_layout()
plt.savefig('saldo_medio_por_descanso_grupos_simplificados.jpg')
plt.show()


#### Resumo numérico por agrupamento simplificado de descanso


In [ ]:

# Resumo numérico usando os mesmos grupos dos gráficos de barras.
# "jogos" usa nunique porque o LeagueGameLog tem duas linhas por partida.
resumo_descanso_oponente = (
    df_barras_desempenho.groupby('REST_COMPARISON_GROUP', observed=False)
    .agg(
        jogos=('GAME_ID', 'nunique'),
        observacoes_time_jogo=('GAME_ID', 'count'),
        vitorias=('WL', 'sum'),
        taxa_vitoria=('WL', 'mean'),
        pontos_marcados=('PTS', 'mean'),
        saldo_medio=('PLUS_MINUS', 'mean'),
        aproveitamento_arremessos=('FG_PCT', 'mean'),
        aproveitamento_3pts=('FG3_PCT', 'mean'),
        rebotes_ofensivos=('OREB', 'mean'),
        assistencias=('AST', 'mean'),
        turnovers=('TOV', 'mean')
    )
    .reset_index()
)

resumo_descanso_oponente['taxa_vitoria'] = resumo_descanso_oponente['taxa_vitoria'] * 100
resumo_descanso_oponente


#### Matrizes: descanso do time x descanso do oponente


In [ ]:
# matriz de taxa de vitória
matriz_vitoria = (
    df.groupby(['REST_DAYS', 'OPP_REST_DAYS'], observed=False)['WL']
    .mean()
    .mul(100)
    .unstack()
    .reindex(index=rest_categories, columns=rest_categories)
)

plt.figure(figsize=(8,6))
sns.heatmap(matriz_vitoria, annot=True, fmt='.1f', cmap='Blues')
plt.title('Taxa de vitória (%) - descanso do time x descanso do oponente', fontsize=14)
plt.xlabel('Descanso do oponente')
plt.ylabel('Descanso do time')
plt.tight_layout()
plt.savefig('heatmap_taxa_vitoria_descanso_oponente.jpg')
plt.show()


In [ ]:
# matriz de saldo médio
matriz_saldo = (
    df.groupby(['REST_DAYS', 'OPP_REST_DAYS'], observed=False)['PLUS_MINUS']
    .mean()
    .unstack()
    .reindex(index=rest_categories, columns=rest_categories)
)

plt.figure(figsize=(8,6))
sns.heatmap(matriz_saldo, annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Saldo médio - descanso do time x descanso do oponente', fontsize=14)
plt.xlabel('Descanso do oponente')
plt.ylabel('Descanso do time')
plt.tight_layout()
plt.savefig('heatmap_saldo_descanso_oponente.jpg')
plt.show()


In [ ]:
# matriz de volume de jogos
matriz_jogos = (
    df.groupby(['REST_DAYS', 'OPP_REST_DAYS'], observed=False)['GAME_ID']
    .count()
    .unstack()
    .reindex(index=rest_categories, columns=rest_categories)
)

plt.figure(figsize=(8,6))
sns.heatmap(matriz_jogos, annot=True, fmt='.0f', cmap='Greens')
plt.title('Número de jogos - descanso do time x descanso do oponente', fontsize=14)
plt.xlabel('Descanso do oponente')
plt.ylabel('Descanso do time')
plt.tight_layout()
plt.savefig('heatmap_jogos_descanso_oponente.jpg')
plt.show()


In [ ]:
plt.figure(figsize=(8,6))
sns.boxplot(x=df.HOME_AWAY.map({0:'Fora de casa', 1: 'Casa'}), y='PLUS_MINUS', data=df, palette='muted')
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.title('Desempenho por local de jogo', fontsize=15, loc='center')
plt.xlabel('')
plt.ylabel('Saldo do jogo', fontsize=13, loc='center')
plt.grid(axis='y', alpha=0.7)
plt.savefig('desempenho_por_local_jogo.jpg')
plt.show()

### Modelagem

In [ ]:
# carregando a base já com descanso do time, descanso do oponente e agrupamento simplificado
# para a modelagem, a variável detalhada REST_MATCHUP continua disponível.
df_modelo = pd.read_csv('nba_descanso_com_oponente_grupos_simplificados.csv')
df_modelo.info()


In [9]:
# vendo a distribuição dos dados da variável target
print(df_modelo.WL.value_counts(normalize=True))
# classes estão bem balanceadas

WL
1    0.500412
0    0.499588
Name: proportion, dtype: float64


In [ ]:
# separando o df em treino e teste

# Nesta versão, a variável categórica principal do modelo é REST_MATCHUP,
# que representa a combinação de descanso do time e do oponente.
# Exemplo: "0 vs 2+" = time em back-to-back contra adversário com 2+ dias de descanso.

colunas_remover = [
    'WL',
    'GAME_ID',
    'TEAM_ABBREVIATION',
    'TEAM_NAME',
    'GAME_DATE',
    'MATCHUP',
    'OPPONENT_ABBREVIATION',
    'PREV_GAME',
    'PLUS_MINUS',
    'FG_PCT',
    'FG3_PCT',
    'FT_PCT',
    'REST_DAYS_NUM',
    'OPP_REST_DAYS_NUM',
    'REST_DAYS',
    'OPP_REST_DAYS',
    'REST_ADVANTAGE',
    'REST_DAYS_NUM_CAT',
    'OPP_REST_DAYS_NUM_CAT',
    'REST_COMPARISON_GROUP'
]

# Separando features e alvo
X = df_modelo.drop(columns=colunas_remover)
y = df_modelo['WL']

# Dividindo treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=y   # garante que a proporção de classes seja igual no treino e no teste
)

# usando one hot encoding para as variáveis categóricas, incluindo REST_MATCHUP
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

# garante que treino e teste tenham exatamente as mesmas colunas após o one hot encoding
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print(f"Treino: {X_train.shape[0]} jogos")
print(f"Teste:  {X_test.shape[0]} jogos")
print()
print(f"Proporção de vitórias no treino: {y_train.mean():.1%}")
print(f"Proporção de vitórias no teste:  {y_test.mean():.1%}")
print()
print(f"Variáveis usadas no modelo: {list(X_train.columns)}")


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix,
                               classification_report, ConfusionMatrixDisplay)

# Treinando o modelo
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

# Previsões
y_pred_lr    = lr.predict(X_test)          # classe: 0 ou 1
y_proba_lr   = lr.predict_proba(X_test)[:, 1]  # probabilidade de ser 1

acuracia_lr = accuracy_score(y_test, y_pred_lr)
print(f"Acurácia da Regressão Logística: {acuracia_lr:.2%}")
print()
print("Relatório completo:")
print(classification_report(y_test, y_pred_lr, target_names=['Fracasso', 'Vitória']))

Acurácia da Regressão Logística: 68.72%

Relatório completo:
              precision    recall  f1-score   support

    Fracasso       0.70      0.66      0.68       243
     Vitória       0.68      0.71      0.69       243

    accuracy                           0.69       486
   macro avg       0.69      0.69      0.69       486
weighted avg       0.69      0.69      0.69       486



In [12]:
coefs = pd.Series(lr.coef_[0], index=X_train.columns)
print(coefs.sort_values(ascending=False))

HOME_AWAY       0.322490
PTS             0.104929
REST_DAYS_2+    0.100872
AST            -0.000270
OREB           -0.001252
REST_DAYS_1    -0.004118
TOV            -0.023862
BACK_TO_BACK   -0.100989
dtype: float64


In [ ]:

# Próximos passos possíveis:
# 1. Testar modelo estatístico com statsmodels/logit usando REST_MATCHUP como variável explicativa detalhada.
# 2. Testar uma versão mais simples do modelo usando REST_COMPARISON_GROUP no lugar de REST_MATCHUP.
# 3. Avaliar se o efeito muda quando o time joga em casa ou fora.
# 4. Controlar por força do time/oponente, por exemplo usando saldo médio, aproveitamento anterior ou seed/conferência.
